# MatCore yabadaba example

This provides a demonstration of building MatCore records using yabadaba.

The yabadaba package provides the base Record class that provides functionality based on schema definitions.  It also has database features to be shown in another Notebook.

The yabadaba_matcore package extends yabadaba by defining Record subclasses associated with the MatCore schemas.

__NOTE__: Use yabadaba version 0.4 from the development repo https://github.com/lmhale99/yabadaba

In [1]:
# Standard Python libraries
from datetime import date
from IPython.display import display, Markdown

import pandas as pd

# Import core yabadaba and extension package
import yabadaba
import yabadaba_matcore

## 1. Check the values of a record

Documentation for a record schema's values can be seen by calling valuedoc().  This generates a markdown string, which can be automatically rendered in a IPython environment like a Jupyter Notebook by setting render=True.

I simply copied the descriptions from the online schema. The descriptions can easily be modified and may contain errors...

In [2]:
yabadaba.valuedoc('matcore', render=True)

# matcore Values

- __creator__ (*list of creator records*): The author who generated the data and institutional affiliation.
    - __name__ (*str*): The name of the author who generated the data.
    - __affiliation__ (*str*): The affiliation of the author who generated the data.
- __title__ (*str*): A single sentence description of the dataset.
- __creation_date__ (*date*): The calendar date when the dataset was created.
- __description__ (*str*): A synopsis of the dataset, its contents, and purpose.
- __disclaimer__ (*str*): A statement of applicability provided by the creator(s) informing users of the intended use and/or limitations of this dataset.
- __material__ (*list of material records*): A description of the chemical composition and structure of a substance in this dataset.
    - __phase__ (*str*): The structure of the material included in the dataset.
    - __description__ (*str*): An explanation of the nature of the material, e.g. a crystal structure designation, a chemical formula for a molecule, etc.
    - __constituent__ (*list of constituent records*): A chemical element included in the material and its concentration.
        - __species__ (*str*): A chemical element included in the material.
        - __concentration__ (*str*): The fraction of the material composed of this element.
    - __microstructure__ (*str*): The arrangement of phases, grains and defects in a material.
- __computation__ (*list of computation records*): A description of a computer simulation used to generate data in the dataset.
    - __method_class__ (*str*): The general category to which the computational technique belongs.
    - __method__ (*str*): The computational materials science (CMS) approach used in the computation.
    - __simulation_conditions__ (*simulation-conditions record*): The interactions between the system being modeled and the rest of the world maintained during the computation.
        - __type__ (*str*): The nature of the simulation being performed, whether equilibrium, nonequilibrium, or nonstandard.
        - __description__ (*str*): Explanatory text clarifying the simulation type selection.
        - __number_of_particles__ (*int*): A count of atoms or other discrete entities comprising the material.
        - __volume__ (*float*): The amount of space occupied by the material.
        - __mass_density__ (*float*): The total number of particles comprising the material divided by the volume it occupies.
        - __cell__ (*floatarray*): The domain occupied by the material in the simulation (if fixed).
        - __cell_reference__ (*floatarray*): The domain occupied by the material in the simulation in configuration relative to which strains are defined.
        - __cell_periodicity__ (*floatarray*): Specification of whether periodic boundary conditions are applied along the cell vector directions. This applies to both cell and cell-reference.
        - __temperature__ (*float*): A measure of the hotness or coldness of an external heat bath in thermal contact with the material.
        - __stress__ (*floatarray*): A uniform force per unit current area, including both normal and shear components, applied to the material during the simulation through contact with an external loading device.
        - __strain__ (*floatarray*): A deformation relative to a reference configuration imposed on the simulation cell expressed in the referential frame (Lagrangian strain tensor).
        - __strain_rate__ (*floatarray*): The derivative with respect to time of a deformation relative to a reference configuration imposed on the simulation cell expressed in the referential frame (Lagrangian strain rate tensor), which is constant throughout the simulation.
        - __heat_flux__ (*floatarray*): The amount of thermal energy transferred to the material per unit area per unit time along a given direction.
        - __temperature_gradient__ (*floatarray*): A physical quantity that describes the rate and direction of maximum temperature change per unit distance.
    - __software__ (*list of constituent records*): Identification of a computer package used to perform the calculations.
        - __name__ (*str*): The name of the software package.
        - __version__ (*str*): The version of the software package.
        - __file__ (*list of file records*): Input or output file for the software package associated with the computation. This includes the input file used to perform the computation, associated data files, and output generated by the software package. Most important are input files to enable reproduction of the dataset.
            - __name__ (*str*): The file name.
            - __description__ (*str*): A brief description of the file and its contents.
            - __contents__ (*str*): The text and/or data contained in the file. Not required in case a link to the file is provided instead.
            - __link__ (*str*): A URI pointing to a permanent location of the file online.
- __citation__ (*list of citation records*): Information that uniquely identifies a source being acknowledged.
    - __reference__ (*str*): Text that uniquely identifies the source of information being cited.
    - __doi__ (*str*): The digital object identifier (DOI) for the source.
    - __link__ (*str*): A URI pointing to a permanent location of the source.
- __funding__ (*list of funding records*): Information about received monetary support or other resources to generate the dataset.
    - __award_title__ (*str*): Name of the grant that provided funding to generate the dataset.
    - __funder__ (*str*): The name of the funding agency that provided money and/or resources to generate the dataset.
    - __award_number__ (*str*): A funder identifier for the grant.
- __related_content__ (*list of related_content records*): Other datasets that are connected to the current one in some manner.
    - __links__ (*strlist*): A list of permanent pointers to related datasets (such as MatCore IDs, DOIs, URIs, etc.)
    - __description__ (*str*): Explanation of the relationship between the related content and the current dataset.
- __provenance__ (*list of provenance records*): History of the dataset, detailing its origins and transformations.
    - __event_type__ (*str*): Description of change made to the dataset.
    - __date__ (*date*): The data when the change was made.
    - __agent__ (*str*): Identity of the entity responsible for the change.
    - __comments__ (*str*): Explanation for the change in provenance.
    - __checksum__ (*str*): A digital fingerprint for the dataset of associated files.
- __matcore_id__ (*str*): An identifier for the dataset.
- __matcore_date__ (*date*): The calendar date when this MatCore document was created.
- __license__ (*str*): A contract defining the terms and conditions under which the dataset can be used.

## 2. Build a record from scratch

You can create a new record of a given style by calling load_record().  

- Values can be set using keyword arguments when calling load_record(), or by setting to similarly named class attributes after creation.
- Subset values can be set when calling load_record() using dict or list of dicts.  They can also be added with add_"subset"() methods after creation.
- The simulation-conditions subset can only have one value. I handled this here by making the subset a record object as well.
- Alternatively, the values in the simulation-conditions subset could be placed in computation instead and the modelpaths changed to show them as 'root.computation.simulation-conditions.NAME'
- User-specified custom elements are also allowed and at least at the surface level behave like the other value objects.

In [3]:
# Create a new record with some properties set 
record = yabadaba.load_record('matcore',
                              title = 'test MatCore yabadaba record',
                              creation_date = date.today(),
                              description = 'This is a test that a MatCore can be constructed using yabadaba.',
                              creator = dict(name='Me', affiliation='the world'),
                              matcore_id = 'dummy-id-5',
                              matcore_date = date.today(),
                              license = 'GPL-3.0-only')

In [4]:
# Assign a property afterwards directly as a class attribute
record.disclaimer = 'This is not a real record, only a tribute'

In [5]:
# Modularly add multi-value subsets with add_"subset" methods.
record.add_material(phase='solid',
                    constituent=[
                        dict(species='Ag', concentration=50),
                        dict(species='Cu', concentration=50)])

In [6]:
record.add_computation(method_class = 'Atomistic', method = 'MD',
                       software = dict(name='LAMMPS'),
                       simulation_conditions = dict(type = 'Equilibrium'))

In [7]:
# Show that simulation_conditions is a record object
record.computation[-1].simulation_conditions

In [8]:
# Values of the simulation-conditions subset can be accessed from the associated object
record.computation[-1].simulation_conditions.type

'Equilibrium'

In [9]:
# Simple values can also be set and transformed 
record.custom = 'Here be my custom value'

## 3. Data transformations

The yabadaba record provides methods for transforming the data between different representations.

__NOTE__: The conversion methods will throw an error if a required value has not been set!

dict/DataFrame representation methods
- metadata() generates a dict containing the simple metadata fields of the record.  This is not meant to be a complete representation, but rather the basis for multi-record DataFrame representations.
- "subset"_df() methods generate a pandas DataFrame of metadata fields for a given subset.

JSON/XML representation methods and attributes
- build_model() generates a dict (DataModelDict) object that is a valid record of the schema, and can be further transformed into XML and JSON representations.
- model is the DataModelDict representation, which exists if the record is loaded from a file or if build_model() was called.
- load_model() reads the record contents from a file (JSON or XML). Or, model contents can be read in when load_record() is called.
- reload_model() updates the record's values based on changes made to model.  This is equivalent to record.load_model(record.model).

In [10]:
record.metadata()

{'name': 'dummy-id-5',
 'creator': [{'name': 'Me', 'affiliation': 'the world'}],
 'title': 'test MatCore yabadaba record',
 'creation_date': datetime.date(2026, 4, 14),
 'description': 'This is a test that a MatCore can be constructed using yabadaba.',
 'disclaimer': 'This is not a real record, only a tribute',
 'material': [{'phase': 'solid',
   'description': None,
   'constituent': [{'species': 'Ag', 'concentration': '50'},
    {'species': 'Cu', 'concentration': '50'}],
   'microstructure': None}],
 'computation': [{'method_class': 'Atomistic',
   'method': 'MD',
   'type': 'Equilibrium',
   'description': None,
   'number_of_particles': None,
   'software': [{'name': 'LAMMPS', 'version': None, 'file': []}]}],
 'citation': [],
 'funding': [],
 'related_content': [],
 'provenance': [],
 'matcore_id': 'dummy-id-5',
 'matcore_date': datetime.date(2026, 4, 14),
 'license': 'GPL-3.0-only',
 'custom': 'Here be my custom value'}

In [11]:
pd.DataFrame([record.metadata()])

,name,creator,title,creation_date,description,disclaimer,material,computation,citation,funding,related_content,provenance,matcore_id,matcore_date,license,custom
0,dummy-id-5,"[{'name': 'Me', 'affiliation': 'the world'}]",test MatCore yabadaba record,2026-04-14,This is a test that a MatCore can be construct...,"This is not a real record, only a tribute","[{'phase': 'solid', 'description': None, 'cons...","[{'method_class': 'Atomistic', 'method': 'MD',...",[],[],[],[],dummy-id-5,2026-04-14,GPL-3.0-only,Here be my custom value


In [12]:
record.creator_df()

,name,affiliation
0,Me,the world


In [13]:
model = record.build_model()
print(model.xml(indent=2))

<?xml version="1.0" encoding="utf-8"?>
<root>
  <creator>
    <name>Me</name>
    <affiliation>the world</affiliation>
  </creator>
  <title>test MatCore yabadaba record</title>
  <creation-date>2026-04-14</creation-date>
  <description>This is a test that a MatCore can be constructed using yabadaba.</description>
  <disclaimer>This is not a real record, only a tribute</disclaimer>
  <material>
    <phase>solid</phase>
    <constituent>
      <species>Ag</species>
      <concentration>50</concentration>
    </constituent>
    <constituent>
      <species>Cu</species>
      <concentration>50</concentration>
    </constituent>
  </material>
  <computation>
    <method-class>Atomistic</method-class>
    <method>MD</method>
    <simulation-conditions>
      <type>Equilibrium</type>
    </simulation-conditions>
    <software>
      <name>LAMMPS</name>
    </software>
  </computation>
  <matcore-id>dummy-id-5</matcore-id>
  <matcore-date>2026-04-14</matcore-date>
  <license>GPL-3.0-only</lic

In [14]:
record2 = yabadaba.load_record('matcore', model=model)

In [15]:
print(record2.model.json(indent=2))

{
  "root": {
    "creator": {
      "name": "Me",
      "affiliation": "the world"
    },
    "title": "test MatCore yabadaba record",
    "creation-date": "2026-04-14",
    "description": "This is a test that a MatCore can be constructed using yabadaba.",
    "disclaimer": "This is not a real record, only a tribute",
    "material": {
      "phase": "solid",
      "constituent": [
        {
          "species": "Ag",
          "concentration": "50"
        },
        {
          "species": "Cu",
          "concentration": "50"
        }
      ]
    },
    "computation": {
      "method-class": "Atomistic",
      "method": "MD",
      "simulation-conditions": {
        "type": "Equilibrium"
      },
      "software": {
        "name": "LAMMPS"
      }
    },
    "matcore-id": "dummy-id-5",
    "matcore-date": "2026-04-14",
    "license": "GPL-3.0-only",
    "custom": "Here be my custom value"
  }
}
